In [2]:
import pandas as pd

In [ ]:
df = pd.read_parquet(r"C:\Users\User\Documents\NUS\Y3S2\DSE3101\dse3101investmentproject\Datasets\13F_clean_files\2023q4_form13f_clean.parquet")
print(df.head())
print(list(df.columns))
print(f"Shape: {df.shape}")

          CIK          FILINGMANAGER_NAME PERIODOFREPORT FILING_DATE  \
0  0000051762  RNC CAPITAL MANAGEMENT LLC     2023-09-30  2023-12-27   
1  0000051762  RNC CAPITAL MANAGEMENT LLC     2023-09-30  2023-12-27   
2  0000051762  RNC CAPITAL MANAGEMENT LLC     2023-09-30  2023-12-27   
3  0000051762  RNC CAPITAL MANAGEMENT LLC     2023-09-30  2023-12-27   
4  0000051762  RNC CAPITAL MANAGEMENT LLC     2023-09-30  2023-12-27   

  SUBMISSIONTYPE  TABLEVALUETOTAL  TABLEENTRYTOTAL ISCONFIDENTIALOMITTED  \
0         13F-HR       1611876095              149                     N   
1         13F-HR       1611876095              149                     N   
2         13F-HR       1611876095              149                     N   
3         13F-HR       1611876095              149                     N   
4         13F-HR       1611876095              149                     N   

                 NAMEOFISSUER      CUSIP     VALUE  SSHPRNAMT    weight  
0         ABBOTT LABORATORIES  00282

In [4]:
df2 = pd.read_parquet(r"C:\Users\User\Documents\NUS\Y3S2\DSE3101\dse3101investmentproject\Datasets\13F_filtered_and_mapped_files\01dec2024-28feb2025_form13f_clean.parquet_final.parquet")
# print(df2.head())
print(list(df2.columns))
print(f"Shape: {df2.shape}")
print(df2[['CIK', 'FILINGMANAGER_NAME', 'PERIODOFREPORT', 'FILING_DATE', "weight", "equity_weight"]].head(10))

['CIK', 'FILINGMANAGER_NAME', 'PERIODOFREPORT', 'FILING_DATE', 'SUBMISSIONTYPE', 'TABLEVALUETOTAL', 'TABLEENTRYTOTAL', 'ISCONFIDENTIALOMITTED', 'NAMEOFISSUER', 'CUSIP', 'VALUE', 'SSHPRNAMT', 'weight', 'ticker', 'security_type', 'equity_portfolio_total', 'equity_weight']
Shape: (370817, 17)
          CIK                    FILINGMANAGER_NAME PERIODOFREPORT  \
0  0001398739  Employees Retirement System of Texas     2024-12-31   
1  0001398739  Employees Retirement System of Texas     2024-12-31   
2  0001398739  Employees Retirement System of Texas     2024-12-31   
3  0001398739  Employees Retirement System of Texas     2024-12-31   
4  0001398739  Employees Retirement System of Texas     2024-12-31   
5  0001398739  Employees Retirement System of Texas     2024-12-31   
6  0001398739  Employees Retirement System of Texas     2024-12-31   
7  0001398739  Employees Retirement System of Texas     2024-12-31   
8  0001398739  Employees Retirement System of Texas     2024-12-31   
9  000139

In [1]:
import duckdb
# Connect to DuckDB (in-memory by default)
con = duckdb.connect()

parquet_file_path = r"C:\Users\User\Documents\NUS\Y3S2\DSE3101\dse3101investmentproject\Datasets\stock_price_data\stock_prices_all.parquet"

# Query the Parquet file directly using DuckDB
con.execute(f"DESCRIBE SELECT * FROM '{parquet_file_path}'").df()

,column_name,column_type,null,key,default,extra
0,date,TIMESTAMP,YES,None,None,None
1,ticker,VARCHAR,YES,None,None,None
2,adj_close,DOUBLE,YES,None,None,None
3,volume,DOUBLE,YES,None,None,None
4,open,DOUBLE,YES,None,None,None
5,high,DOUBLE,YES,None,None,None
6,low,DOUBLE,YES,None,None,None
7,close,DOUBLE,YES,None,None,None
8,year,INTEGER,YES,None,None,None


In [ ]:
# print count of unique tickers 
result = con.execute(f"""
                     SELECT COUNT(DISTINCT ticker) AS unique_tickers
                     FROM '{parquet_file_path}'
                     """).df()
print(result)

   unique_tickers
0            5779


In [3]:
result = con.execute(f"""
    SELECT ticker, COUNT(*) AS row_count
    FROM '{parquet_file_path}'
    GROUP BY ticker
    ORDER BY row_count DESC
""").df()
print(result)

     ticker  row_count
0      FWRD       3315
1       WLK       3315
2       FLO       3315
3       CNS       3315
4      BMCS       3315
...     ...        ...
5774   OGNT          8
5775   NXPS          7
5776   GNPR          6
5777   GITH          2
5778    CTH          1

[5779 rows x 2 columns]


In [4]:
result = con.execute(f"""
    SELECT
        ticker,
        MIN(close)  AS min_price,
        MAX(close)  AS max_price,
        AVG(close)  AS avg_price,
        STDDEV(close) AS stddev_price
    FROM '{parquet_file_path}'
    GROUP BY ticker
    ORDER BY ticker
""").df()
print(result)


     ticker  min_price    max_price   avg_price  stddev_price
0         A  28.984262   179.279999   86.888825     42.091785
1        AA   5.480000    95.059998   33.250727     13.289325
2      AABB   0.000470     0.513022    0.028533      0.045279
3     AABVF   0.010000     0.370000    0.087483      0.066287
4     AACTF   0.006000     0.590000    0.069091      0.086299
...     ...        ...          ...         ...           ...
5774   ZVRA   2.752000   378.079987   48.819918     77.104463
5775   ZVSA   0.115000  7875.000000  804.934423   1405.002018
5776    ZWS   7.061657    52.779999   19.800669     10.761249
5777   ZYME   4.650000    56.810001   18.170062     11.995240
5778  ZYXIQ   0.030000    25.981817    5.386224      5.378521

[5779 rows x 5 columns]
